In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
pip install transformers==4.30.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 5.1 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.0.0rc2
    Uninstalling huggingface-hub-1.0.0rc2:
      Successfully uninstalled huggingface-hub-1.0.0rc2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of

In [3]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch import nn
from transformers import CLIPProcessor, CLIPModel
from transformers import CLIPImageProcessor, AutoTokenizer

from sklearn.model_selection import train_test_split
from tqdm import tqdm

# --- Configuration ---
# Set device to GPU if available, otherwise CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Hyperparameters
EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

# Dataset path
DATASET_PATH = "/kaggle/input/ai-generated-images-vs-real-images/"

2025-11-03 16:13:13.068982: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762186393.269197      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762186393.326478      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using device: cuda


In [4]:
from huggingface_hub import login
import os

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    
    print("Logging into Hugging Face using Kaggle Secret...")
    login(token=hf_token)
    print("Login successful.")

except ImportError:
    print("Kaggle secrets not found (running locally?). Make sure HF_TOKEN is set in your environment.")
    if "HF_TOKEN" in os.environ:
        login(token=os.environ["HF_TOKEN"])

except Exception as e:
    print(f"Could not log into Hugging Face: {e}")
    pass

Logging into Hugging Face using Kaggle Secret...
Login successful.


In [5]:
# --- Data Preparation ---

# Get the base path for the dataset
DATASET_PATH = "/kaggle/input/ai-generated-images-vs-real-images/"

# --- CORRECTED: Point to the 'train' and 'test' directories ---
train_paths = []
val_paths = []

# Populate training paths
train_real_path = os.path.join(DATASET_PATH, 'train/real')
for img_name in os.listdir(train_real_path):
    train_paths.append((os.path.join(train_real_path, img_name), 0)) # Label 0 for Real

train_fake_path = os.path.join(DATASET_PATH, 'train/fake')
for img_name in os.listdir(train_fake_path):
    train_paths.append((os.path.join(train_fake_path, img_name), 1)) # Label 1 for Fake

# Populate validation paths from the 'test' directory
val_real_path = os.path.join(DATASET_PATH, 'test/real')
for img_name in os.listdir(val_real_path):
    val_paths.append((os.path.join(val_real_path, img_name), 0)) # Label 0 for Real

val_fake_path = os.path.join(DATASET_PATH, 'test/fake')
for img_name in os.listdir(val_fake_path):
    val_paths.append((os.path.join(val_fake_path, img_name), 1)) # Label 1 for Fake

# Shuffle the datasets for randomness
random.shuffle(train_paths)
random.shuffle(val_paths)

print(f"Training samples: {len(train_paths)}")
print(f"Validation samples: {len(val_paths)}")


# --- Custom PyTorch Dataset ---
class ImageDataset(Dataset):
    def __init__(self, image_paths, processor):
        self.image_paths = image_paths
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path, label = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert("RGB")
            # Process the image using the CLIP processor
            processed_image = self.processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)
            return processed_image, torch.tensor(label, dtype=torch.float32)
        except Exception as e:
            # Handle potential corrupted images
            print(f"Skipping corrupted image: {img_path}, error: {e}")
            # Return the first image as a placeholder to avoid crashing the loader
            return self.__getitem__(0)

# Load the CLIP processor
clip_model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(clip_model_name)

# Create Dataset and DataLoader instances
train_dataset = ImageDataset(train_paths, processor)
val_dataset = ImageDataset(val_paths, processor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

Training samples: 48000
Validation samples: 12000


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [6]:
# pip install transformers==4.30.2

In [7]:
# # --- CORRECTED CODE to handle library updates ---
# # Load the CLIP processor's components separately
# from transformers import CLIPImageProcessor, CLIPTokenizer, CLIPProcessor

# clip_model_name = "openai/clip-vit-base-patch32"

# # Use the specific CLIPTokenizer class to avoid the error
# tokenizer = CLIPTokenizer.from_pretrained(clip_model_name)
# image_processor = CLIPImageProcessor.from_pretrained(clip_model_name)

# # Manually combine them into a CLIPProcessor instance
# processor = CLIPProcessor(image_processor=image_processor, tokenizer=tokenizer)

# Now you can use the processor as intended
print("Processor loaded successfully!")
print(processor)

Processor loaded successfully!
CLIPProcessor:
- image_processor: CLIPImageProcessor {
  "crop_size": {
    "height": 224,
    "width": 224
  },
  "do_center_crop": true,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "feature_extractor_type": "CLIPFeatureExtractor",
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_processor_type": "CLIPImageProcessor",
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "shortest_edge": 224
  }
}

- tokenizer: CLIPTokenizerFast(name_or_path='openai/clip-vit-base-patch32', vocab_size=49408, model_max_length=77, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': AddedToken("<|startoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True), 'eos_token': AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalize

In [8]:
# --- Model Architecture ---
class CLIPImageClassifier(nn.Module):
    def __init__(self, clip_model_name="openai/clip-vit-base-patch32"):
        super(CLIPImageClassifier, self).__init__()
        # Load the pretrained CLIP model
        self.clip = CLIPModel.from_pretrained(clip_model_name)

        # Freeze all the parameters in the CLIP model
        for param in self.clip.parameters():
            param.requires_grad = False

        # Define a custom classification head
        self.classifier = nn.Sequential(
            nn.Linear(self.clip.config.vision_config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
            nn.Sigmoid()  # Sigmoid for binary classification
        )

    def forward(self, pixel_values):
        # Get the image features from the CLIP vision model
        vision_outputs = self.clip.vision_model(pixel_values=pixel_values)
        # The pooler_output gives a summary feature vector for the image
        image_features = vision_outputs.pooler_output
        
        # Pass the features through our custom classifier
        return self.classifier(image_features)

# Instantiate the model and move it to the GPU
model = CLIPImageClassifier().to(DEVICE)

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [9]:
!nvidia-smi

Mon Nov  3 16:13:30 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             26W /   70W |     733MiB /  15360MiB |     19%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
# --- Training ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

model.to(DEVICE)

# Optimizer and Loss Function
# We only pass the parameters of the trainable classifier head to the optimizer
optimizer = Adam(model.classifier.parameters(), lr=LEARNING_RATE)
criterion = nn.BCELoss() # Binary Cross-Entropy Loss for binary classification

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for images, labels in tqdm(dataloader, desc="Training"):
        images, labels = images.to(device), labels.to(device).unsqueeze(1)
        
        # Zero the gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        
        # Calculate loss
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validating"):
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            # Calculate accuracy
            predicted = (outputs > 0.5).float()
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)
            
    accuracy = correct_predictions / total_samples
    return total_loss / len(dataloader), accuracy

best_val_accuracy = 0.0  # Initialize a tracker for the best accuracy
SAVE_PATH = "/kaggle/working/best_clip_finetuned_classifier.pth" # Path for the best model

# --- Main Loop ---
for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_accuracy = validate(model, val_loader, criterion, DEVICE)
    
    print(f"Epoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Validation Loss: {val_loss:.4f}")
    print(f"  Validation Accuracy: {val_accuracy:.4f}")

    # --- Check if the current model is the best one and save it ---
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        # Save the state dictionary of the best model
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"🎉 New best model saved! Accuracy: {val_accuracy:.4f} at Epoch {epoch+1}")
        print(f"   Model saved to {SAVE_PATH}")


# Save the trained model (optional)
# torch.save(model.state_dict(), 'clip_finetuned_classifier.pth')
print("\nTraining complete.")
print(f"The best model with accuracy {best_val_accuracy:.4f} is saved at {SAVE_PATH}")


Using device: cuda

--- Epoch 1/3 ---


Training:   0%|          | 7/1500 [00:35<1:58:47,  4.77s/it]

Skipping corrupted image: /kaggle/input/ai-generated-images-vs-real-images/train/real/12094.jpg, error: image file is truncated (6 bytes not processed)


Training:   4%|▍         | 58/1500 [04:26<2:18:10,  5.75s/it]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Training:  27%|██▋       | 410/1500 [31:30<1:17:30,  4.27s/it]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107184040 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Training:  47%|████▋     | 706/1500 [54:06<1:05:29,  4.95s/it]

Skipping corrupted image: /kaggle/input/ai-generated-images-vs-real-images/train/real/15963.jpg, error: image file is truncated (2 bytes not processed)


...

Skipping corrupted image: /kaggle/input/ai-generated-images-vs-real-images/test/real/5325.jpg, error: image file is truncated (3 bytes not processed)


Validating:  65%|██████▍   | 242/375 [17:50<09:25,  4.25s/it]

Skipping corrupted image: /kaggle/input/ai-generated-images-vs-real-images/test/real/5879.jpg, error: image file is truncated


Validating: 100%|██████████| 375/375 [27:13<00:00,  4.36s/it]


Epoch 3 Summary:
  Train Loss: 0.1154
  Validation Loss: 0.1089
  Validation Accuracy: 0.9591
🎉 New best model saved! Accuracy: 0.9591 at Epoch 3
   Model saved to /kaggle/working/best_clip_finetuned_classifier.pth

Training complete.
The best model with accuracy 0.9591 is saved at /kaggle/working/best_clip_finetuned_classifier.pth
